# Dataset Generation: Granular HDX-Base with Round-Wise CIRV Alignment

This notebook builds a high-granularity dataset using the **HDX CERF Allocations** as the base table. 

### Key Logic:
1. **Full Data Retention**: All columns from the HDX allocations are preserved (agency, sector, project details, etc.).
2. **Strict Temporal Alignment**: Every project is assigned to Round I (Jan-Jun) or Round II (Jul-Dec) based on the signature date.
3. **Enrichment**: Each HDX row is enriched with the three requested CIRV indicators (Excl. Inform, Incl. Inform, Adjusted) corresponding to its Round.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# Paths
CIRV_DIR = Path('../initial_data/cirv')
HDX_PATH = Path('../initial_data/hdx_cerf_allocations.csv')
OUTPUT_PATH = Path('../initial_data/unified_cerf_2022_2024.csv')

def standardize_iso3(val):
    if not isinstance(val, str): return val
    return val.strip().upper()

def get_round(month):
    return 'I' if month <= 6 else 'II'

## 1. Process HDX Funding (Base Table)
Assign Rounds to each project without aggregating, keeping all original columns.

In [2]:
hdx = pd.read_csv(HDX_PATH)

# Filter for UFE window and 2022-2024
hdx_ufe = hdx[(hdx['windowFullName'] == 'Underfunded Emergencies') & 
              (hdx['year'].between(2022, 2024))].copy()

# Assign Round based on month
hdx_ufe['date'] = pd.to_datetime(hdx_ufe['dateUSGSignature'])
hdx_ufe['Round'] = hdx_ufe['date'].dt.month.apply(get_round)
hdx_ufe['ISO3_Standard'] = hdx_ufe['countryCode'].apply(standardize_iso3)

# Rename year to Year for merging
hdx_ufe.rename(columns={'year': 'Year'}, inplace=True)

print(f"Processing {len(hdx_ufe)} granular project allocations from HDX.")
hdx_ufe.head()

Processing 396 granular project allocations from HDX.


,agencyName,continentName,countryCode,countryName,dateUSGSignature,emergencyTypeName,projectCode,projectID,projectTitle,regionName,...,totalAmountApproved,windowFullName,Year,projectsectors,projectclusters,projectgroupings,projectcapcodes,date,Round,ISO3_Standard
7083,Food and Agriculture Organization,Africa,NER,Niger,2022-01-31,Drought,22-UF-FAO-003,4084,Appui aux moyens de subsistance des ménages vu...,Western Africa,...,2600000.0,Underfunded Emergencies,2022,Agriculture,Agriculture,NaN,NaN,2022-01-31,I,NER
7084,United Nations Children’s Fund,Africa,NER,Niger,2022-02-02,Drought,22-UF-CEF-004,4085,Prévention et prise en charge de l’émaciation ...,Western Africa,...,1500000.0,Underfunded Emergencies,2022,Nutrition,Nutrition,NaN,NaN,2022-02-02,I,NER
7085,United Nations Office for Project Services,Africa,NER,Niger,2022-01-31,Drought,22-UF-OPS-001,4086,Sensibilisation des populations les plus vulné...,Western Africa,...,298571.0,Underfunded Emergencies,2022,Protection,Protection,NaN,NaN,2022-01-31,I,NER
7086,World Food Programme,Africa,NER,Niger,2022-02-01,Drought,22-UF-WFP-005,4087,Assistance alimentaire et nutritionnelle aux p...,Western Africa,...,5200088.0,Underfunded Emergencies,2022,"Food Assistance, Nutrition","Food Assistance, Nutrition",NaN,NaN,2022-02-01,I,NER
7087,World Health Organization,Africa,NER,Niger,2022-02-02,Drought,22-UF-WHO-003,4088,Assistance médico-sanitaire aux populations af...,Western Africa,...,400000.0,Underfunded Emergencies,2022,Health,Health,NaN,NaN,2022-02-02,I,NER


## 2. Process CIRV Indicators
Extract specific scores from round-based Excel files.

In [3]:
cirv_frames = []

indicator_map = {
    'CIRV - CERF Index for Risk and Vulnerability (without Inform Severity)': 'CIRV (excluding Inform)',
    'CIRV (Incl Inform Severity)': 'CIRV (including Inform)',
    'CIRV (Adjusted)': 'CIRV (Adjusted)'
}

for excel_file in sorted(CIRV_DIR.glob('*.xlsx')):
    stem = excel_file.stem
    match = re.search(r'(\d{4})-(I+)', stem)
    if not match: continue
    year = int(match.group(1))
    round_lbl = match.group(2)
    
    if year < 2022 or year > 2024: continue
    
    # Find the data sheet
    wb = pd.ExcelFile(excel_file)
    sheet_name = [s for s in wb.sheet_names if 'All Data' in s]
    if not sheet_name: continue
    
    df = pd.read_excel(excel_file, sheet_name=sheet_name[0], header=2)
    df.columns = [str(c).strip() for c in df.columns]
    
    # Keep only relevant columns
    cols_to_extract = ['ISO3 Country Code'] + list(indicator_map.keys())
    existing_cols = [c for c in cols_to_extract if c in df.columns]
    
    df_subset = df[existing_cols].copy()
    
    # Rename indicators for consistency
    df_subset.rename(columns=indicator_map, inplace=True)
    
    # Metadata
    df_subset['Year'] = year
    df_subset['Round'] = round_lbl
    df_subset['ISO3_Standard'] = df_subset['ISO3 Country Code'].apply(standardize_iso3)
    
    # Drop rows without ISO3
    df_subset = df_subset.dropna(subset=['ISO3_Standard'])
    
    cirv_frames.append(df_subset)

cirv_full = pd.concat(cirv_frames, ignore_index=True)

# Ensure only one row per ISO3/Year/Round (take first if duplicates exist in sheet)
cirv_full = cirv_full.drop_duplicates(subset=['Year', 'Round', 'ISO3_Standard'])

print(f"Extracted {len(cirv_full)} indicator sets from {len(cirv_frames)} Excel files.")
cirv_full.head()

/Users/leonardkuch/Documents/Projects/Coding/datathon/analytics_datathon_2026/.venv/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Sparkline Group extension is not supported and will be removed
  warn(msg)


/Users/leonardkuch/Documents/Projects/Coding/datathon/analytics_datathon_2026/.venv/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Sparkline Group extension is not supported and will be removed
  warn(msg)


/Users/leonardkuch/Documents/Projects/Coding/datathon/analytics_datathon_2026/.venv/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Sparkline Group extension is not supported and will be removed
  warn(msg)


/Users/leonardkuch/Documents/Projects/Coding/datathon/analytics_datathon_2026/.venv/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Sparkline Group extension is not supported and will be removed
  warn(msg)


Extracted 906 indicator sets from 6 Excel files.


/Users/leonardkuch/Documents/Projects/Coding/datathon/analytics_datathon_2026/.venv/lib/python3.9/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Sparkline Group extension is not supported and will be removed
  warn(msg)


,ISO3 Country Code,CIRV (excluding Inform),CIRV (including Inform),CIRV (Adjusted),Year,Round,ISO3_Standard
0,TEST,100.000000,100.000000,100.000000,2022,I,TEST
1,AFG,78.047778,83.365185,83.365185,2022,I,AFG
2,ALB,14.000000,9.333333,14.000000,2022,I,ALB
3,DZA,21.975556,14.650370,21.975556,2022,I,DZA
4,AGO,43.116667,28.744444,43.116667,2022,I,AGO


## 3. Unified Join
Left join HDX base table with CIRV indicators. Every HDX project will now have the round-specific indicators attached.

In [4]:
# Left join on Year, Round, and ISO3
final_df = pd.merge(
    hdx_ufe,
    cirv_full.drop(columns=['ISO3 Country Code']),
    on=['Year', 'Round', 'ISO3_Standard'],
    how='left'
)

# Reorder columns to show Round and indicators near the front
cols = list(final_df.columns)
front_cols = ['Year', 'Round', 'ISO3_Standard', 'countryName', 
              'CIRV (excluding Inform)', 'CIRV (including Inform)', 'CIRV (Adjusted)']
other_cols = [c for c in cols if c not in front_cols]
final_df = final_df[front_cols + other_cols]

print(f"Final Granular Dataset Shape: {final_df.shape}")
final_df.head(20)

Final Granular Dataset Shape: (396, 24)


,Year,Round,ISO3_Standard,countryName,CIRV (excluding Inform),CIRV (including Inform),CIRV (Adjusted),agencyName,continentName,countryCode,...,projectTitle,regionName,tableName,totalAmountApproved,windowFullName,projectsectors,projectclusters,projectgroupings,projectcapcodes,date
0,2022,I,NER,Niger,46.333333,56.888889,56.888889,Food and Agriculture Organization,Africa,NER,...,Appui aux moyens de subsistance des ménages vu...,Western Africa,P,2600000.0,Underfunded Emergencies,Agriculture,Agriculture,NaN,NaN,2022-01-31
1,2022,I,NER,Niger,46.333333,56.888889,56.888889,United Nations Children’s Fund,Africa,NER,...,Prévention et prise en charge de l’émaciation ...,Western Africa,P,1500000.0,Underfunded Emergencies,Nutrition,Nutrition,NaN,NaN,2022-02-02
2,2022,I,NER,Niger,46.333333,56.888889,56.888889,United Nations Office for Project Services,Africa,NER,...,Sensibilisation des populations les plus vulné...,Western Africa,P,298571.0,Underfunded Emergencies,Protection,Protection,NaN,NaN,2022-01-31
3,2022,I,NER,Niger,46.333333,56.888889,56.888889,World Food Programme,Africa,NER,...,Assistance alimentaire et nutritionnelle aux p...,Western Africa,P,5200088.0,Underfunded Emergencies,"Food Assistance, Nutrition","Food Assistance, Nutrition",NaN,NaN,2022-02-01
4,2022,I,NER,Niger,46.333333,56.888889,56.888889,World Health Organization,Africa,NER,...,Assistance médico-sanitaire aux populations af...,Western Africa,P,400000.0,Underfunded Emergencies,Health,Health,NaN,NaN,2022-02-02
5,2022,I,TCD,Chad,55.722222,65.148148,65.148148,United Nations Children’s Fund,Africa,TCD,...,"Réponse intégrée d'urgence en Eau, Hygiène et ...",Middle Africa,P,3274940.0,Underfunded Emergencies,"Nutrition, Water, Sanitation and Hygiene, Prot...","Nutrition, Water, Sanitation and Hygiene, Prot...",NaN,NaN,2022-03-04
6,2022,I,TCD,Chad,55.722222,65.148148,65.148148,World Health Organization,Africa,TCD,...,Intervention médicale pour réduire la morbidit...,Middle Africa,P,674433.0,Underfunded Emergencies,"Health, Nutrition","Health, Nutrition",NaN,NaN,2022-02-24
7,2022,I,TCD,Chad,55.722222,65.148148,65.148148,Food and Agriculture Organization,Africa,TCD,...,Reconstitution d’urgence des moyens d’existenc...,Middle Africa,P,500001.0,Underfunded Emergencies,Agriculture,Agriculture,NaN,NaN,2022-03-01
8,2022,I,TCD,Chad,55.722222,65.148148,65.148148,United Nations High Commissioner for Refugees,Africa,TCD,...,Protection et assistance d’urgence aux populat...,Middle Africa,P,1550000.0,Underfunded Emergencies,"Protection, Shelter and Non-Food Items, Camp C...","Protection, Shelter and Non-Food Items, Camp C...",NaN,NaN,2022-03-04
9,2022,I,TCD,Chad,55.722222,65.148148,65.148148,World Food Programme,Africa,TCD,...,Assistance alimentaire et nutritionnelle pour ...,Middle Africa,P,2000000.0,Underfunded Emergencies,"Food Assistance, Nutrition","Food Assistance, Nutrition",NaN,NaN,2022-02-22


## 4. Export

In [5]:
final_df.to_csv(OUTPUT_PATH, index=False)
print(f"Granular unified dataset successfully saved to {OUTPUT_PATH}")

Granular unified dataset successfully saved to ../initial_data/unified_cerf_2022_2024.csv
